# Gerenciador Spark Base

Módulo universal e obrigatório para qualquer projeto executado em Apache Spark.
Disponibiliza:
- Configurações e leitura de variáveis de ambiente.
- Sistema unificado de logs e telemetria (`GerenciadorLogs`, `logs`, `logs_rotina`).
- Tratamento e registro padronizado de exceções em rotinas.
- Governança e anonimização de dados LGPD (`anonimizar_coluna`, `publicar_tabela_anonimizada`).
- Utilitários globais do Apache Spark.


## 1. Imports e Tipagens Base


In [ ]:
%%spark

import os
import re
import sys
import time
import datetime
import decimal
import traceback
from typing import Dict, List, Optional, Any, Union, Tuple, Set

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, DataType


## 2. Variáveis de Ambiente e Gestão de Erros


In [ ]:
%%spark

def obter_variavel_ambiente(
    nome_variavel: str,
    obrigatoria: bool = True,
    padrao: Optional[str] = None,
) -> Optional[str]:
    """
    Obtém o valor de uma variável de ambiente do sistema/Spark.
    """
    valor = os.environ.get(nome_variavel)
    if valor is None or not str(valor).strip():
        if obrigatoria:
            raise ValueError(f"Variável de ambiente obrigatória não informada: '{nome_variavel}'")
        return padrao
    return str(valor).strip()


def registrar_erro_rotina(etapa: str, excecao: Exception, logger=None) -> None:
    """
    Registra no log padrão a ocorrência de erro em uma etapa da rotina.
    """
    mensagem = f"[ERRO NA ROTINA] Falha na etapa '{etapa}'. Tipo: {type(excecao).__name__} | Mensagem: {str(excecao)}"
    log_instancia = logger or logs_rotina or logs
    if log_instancia and hasattr(log_instancia, "erro"):
        log_instancia.erro(mensagem)
        log_instancia.erro(traceback.format_exc())
    else:
        print(mensagem)
        print(traceback.format_exc())


## 3. Sistema Unificado de Registro e Rastreabilidade (Logging)


In [ ]:
%%spark

class GerenciadorLogs:
    """
    Gerenciador de logs formatados para console Spark e Jupyter Notebooks.
    """
    def __init__(self, nome: str = "PIPELINE") -> None:
        self.nome = nome

    def _prefixo(self) -> str:
        data_hora = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        return f"[{data_hora}][{self.nome}]"

    def subetapa(self, sufixo: str) -> "GerenciadorLogs":
        return GerenciadorLogs(f"{self.nome}.{sufixo}")

    def info(self, mensagem: str) -> None:
        print(f"{self._prefixo()}[INFO] {mensagem}")
        sys.stdout.flush()

    def sucesso(self, mensagem: str) -> None:
        print(f"{self._prefixo()}[OK] {mensagem}")
        sys.stdout.flush()

    def aviso(self, mensagem: str) -> None:
        print(f"{self._prefixo()}[AVISO] {mensagem}")
        sys.stdout.flush()

    def erro(self, mensagem: str) -> None:
        print(f"{self._prefixo()}[ERRO] {mensagem}")
        sys.stdout.flush()

    def inspecionar(self, valor: Any, titulo: Optional[str] = None) -> None:
        rotulo = f" {titulo}" if titulo else ""
        print(f"{self._prefixo()}[OBJETO{rotulo}] {valor}")
        sys.stdout.flush()

    def exibir_dataframe(
        self,
        df: DataFrame,
        qtd_linhas: int = 20,
        truncar: bool = True,
        titulo: Optional[str] = None,
    ) -> None:
        if titulo:
            print(f"{self._prefixo()}[DATAFRAME: {titulo}]")
        if df is not None:
            df.show(n=int(qtd_linhas), truncate=truncar)
        sys.stdout.flush()

    def exibir_schema(self, df: DataFrame, titulo: Optional[str] = None) -> None:
        if titulo:
            print(f"{self._prefixo()}[SCHEMA: {titulo}]")
        if df is not None:
            df.printSchema()
        sys.stdout.flush()


class GerenciadorLogsNulo:
    """
    Registrador silencioso para execuções em segundo plano sem saída de console.
    """
    def subetapa(self, sufixo: str) -> "GerenciadorLogsNulo":
        return self
    def info(self, mensagem: str) -> None:
        pass
    def sucesso(self, mensagem: str) -> None:
        pass
    def aviso(self, mensagem: str) -> None:
        pass
    def erro(self, mensagem: str) -> None:
        pass
    def inspecionar(self, valor: Any, titulo: Optional[str] = None) -> None:
        pass
    def exibir_dataframe(self, df: DataFrame, qtd_linhas: int = 20, truncar: bool = True, titulo: Optional[str] = None) -> None:
        pass
    def exibir_schema(self, df: DataFrame, titulo: Optional[str] = None) -> None:
        pass


def criar_gerenciador_logs(nome: str = "PIPELINE", ativo: bool = True) -> Union[GerenciadorLogs, GerenciadorLogsNulo]:
    usar_logs_env = os.environ.get("USE_LOGS", "true").lower() in ("true", "1", "s", "sim", "yes")
    if ativo and usar_logs_env:
        return GerenciadorLogs(nome=nome)
    return GerenciadorLogsNulo()


logs = criar_gerenciador_logs("PIPELINE")
logs_rotina = criar_gerenciador_logs("ROTINA")


## 4. Governança e Anonimização de Dados (LGPD)


In [ ]:
%%spark

def anonimizar_coluna(
    df: DataFrame,
    coluna_origem: str = "CD_CLI",
    algoritmo: str = "md5",
    prefixo_anonimizado: str = "DA_",
) -> DataFrame:
    """
    Gera uma cópia do DataFrame com a coluna identificadora anonimizada via hash criptográfico.
    """
    if coluna_origem not in df.columns:
        raise ValueError(f"Coluna de origem '{coluna_origem}' não encontrada no DataFrame.")
    
    coluna_anonimizada = f"{prefixo_anonimizado}{coluna_origem}"
    
    if algoritmo.lower() == "sha256":
        funcao_hash = F.sha2(F.col(coluna_origem).cast("string"), 256)
    else:
        funcao_hash = F.md5(F.col(coluna_origem).cast("string"))
        
    df_transformado = (
        df.withColumn(coluna_anonimizada, funcao_hash)
        .drop(coluna_origem)
    )
    
    colunas_ordenadas = [
        coluna_anonimizada if c == coluna_origem else c
        for c in df.columns
    ]
    return df_transformado.select(*colunas_ordenadas)


def publicar_tabela_anonimizada(
    df: DataFrame,
    database: str,
    tabela: str,
    modo_gravacao: str = "append",
    coluna_origem: str = "CD_CLI",
    algoritmo: str = "md5",
) -> None:
    """
    Anonimiza e publica um DataFrame no Hive/Metastore em tabela com prefixo ANDO_.
    """
    if not database or not tabela:
        raise ValueError("Database e tabela de destino devem ser informados.")
        
    tabela_destino = f"{database}.ANDO_{tabela}"
    coluna_anonimizada = f"DA_{coluna_origem}"
    
    if coluna_origem not in df.columns:
        print(f"[ALERTA] Coluna de origem '{coluna_origem}' ausente. Publicação abortada para '{tabela_destino}'.")
        return
        
    try:
        df_anonimizado = anonimizar_coluna(df, coluna_origem=coluna_origem, algoritmo=algoritmo)
        (
            df_anonimizado.write
            .mode(modo_gravacao)
            .insertInto(tabela_destino)
        )
        print(f"[OK] Publicação concluída com sucesso: '{tabela_destino}' com '{coluna_anonimizada}' (CHAR(32)).")
    except Exception as exc:
        print(f"[ERRO] Falha ao gravar tabela anonimizada '{tabela_destino}': {exc}")
        raise


## 5. Utilitários Globais do Spark\n\nFunções de apoio a pipelines e sessões interativas.


In [ ]:
%%spark

def verificar_tabela_existe(nome_completo_tabela: str, sessao_spark: Optional[SparkSession] = None) -> bool:
    """
    Verifica se uma tabela ou view existe no metastore/catálogo Spark sem disparar erros.
    """
    spk = sessao_spark or spark
    try:
        spk.table(nome_completo_tabela)
        return True
    except Exception:
        return False


def obter_contagem_particoes(df: DataFrame) -> int:
    """
    Retorna o número atual de partições físicas do DataFrame RDD.
    """
    if df is None:
        return 0
    return df.rdd.getNumPartitions()


def coalescer_seguro(df: DataFrame, max_particoes: int = 1) -> DataFrame:
    """
    Reduz o número de partições de forma segura se a contagem atual for maior que max_particoes.
    """
    if df is None:
        return df
    qtd_atual = df.rdd.getNumPartitions()
    if qtd_atual > max_particoes:
        return df.coalesce(int(max_particoes))
    return df

print("Módulo gerenciador_spark carregado com sucesso.")
